In [ ]:
# Installation of ifcopenshell
!pip install ifcopenshell

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 MB 16.8 MB/s eta 0:00:00


In [ ]:
# Reads attributes from an excel file an writes them into the ifc file. Useful to attach attribute to
# IfcBuilding or IfcBuildingStorey, which is not natively possible in revit.
# Version 0.1 - 26.03.2026 - IFC4

import ifcopenshell
import ifcopenshell.util.element
import pandas as pd
import uuid

# --- Configuration ---
ifc_path = '0004901726100001 A-2M GAGP 999-00 Testmodell Architektur.ifc'
excel_path = 'mapping_data.xlsx'
output_path = 'updated_model.ifc'

# --- Load Data ---
ifc_file = ifcopenshell.open(ifc_path)
df = pd.read_excel(excel_path)

def get_or_create_pset(element, pset_name, ifc_file):
    for rel in getattr(element, 'IsDefinedBy', []):
        if rel.is_a('IfcRelDefinesByProperties'):
            definition = rel.RelatingPropertyDefinition
            if definition.is_a('IfcPropertySet') and definition.Name == pset_name:
                # Check if shared; if so, we must not reuse it to avoid modifying other elements
                if len(getattr(definition, 'HasAssignments', [])) > 1:
                    break
                return definition

    # Always create a NEW pset if no unique one is found
    new_pset = ifc_file.create_entity(
        'IfcPropertySet',
        GlobalId=ifcopenshell.guid.new(),
        Name=pset_name,
        HasProperties=[]
    )

    ifc_file.create_entity(
        'IfcRelDefinesByProperties',
        GlobalId=ifcopenshell.guid.new(),
        RelatedObjects=[element],
        RelatingPropertyDefinition=new_pset
    )
    return new_pset

def update_property(pset, prop_name, value, data_type, ifc_file):
    data_type = str(data_type).strip() if not pd.isna(data_type) else 'IfcText'
    dt_lower = data_type.lower()
    raw_val_str = str(value).lower().strip()

    if dt_lower == 'ifclogical':
        if raw_val_str in ['true', '1', 't', '.t.', 'wahr', '1.0']:
            val_obj = ifc_file.create_entity('IfcLogical', True)
        elif raw_val_str in ['false', '0', 'f', '.f.', 'falsch', '0.0']:
            val_obj = ifc_file.create_entity('IfcLogical', False)
        else:
            val_obj = ifc_file.create_entity('IfcLogical', None)
    elif dt_lower == 'ifcboolean':
        bool_val = raw_val_str in ['true', '1', 't', 'wahr', '1.0']
        val_obj = ifc_file.create_entity('IfcBoolean', bool_val)
    elif 'integer' in dt_lower or 'countmeasure' in dt_lower:
        val_obj = ifc_file.create_entity(data_type, int(float(str(value).replace(',', '.'))))
    elif any(x in dt_lower for x in ['measure', 'length', 'area', 'volume', 'real']):
        val_obj = ifc_file.create_entity(data_type, float(str(value).replace(',', '.')))
    else:
        val_obj = ifc_file.create_entity(data_type, str(value))

    # Replace existing property
    current_props = list(pset.HasProperties)
    new_props = [p for p in current_props if p.Name != prop_name]

    new_prop = ifc_file.create_entity('IfcPropertySingleValue', Name=prop_name, NominalValue=val_obj)
    new_props.append(new_prop)
    pset.HasProperties = new_props

print(f"Processing {len(df)} entries...")
for _, row in df.iterrows():
    element = ifc_file.by_guid(row['ifcGUID'])
    if element:
        target_pset = get_or_create_pset(element, row['pset'], ifc_file)
        update_property(target_pset, row['attribute name'], row['value'], row['data type'], ifc_file)

ifc_file.write(output_path)
print(f"Done. Saved to {output_path}")

Processing 33 entries...
Done. Saved to updated_model.ifc
